<a href="https://colab.research.google.com/github/Rayan1156/phishing_email_detection/blob/main/DistilBERT_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers

import torch
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")


GPU available: True
GPU: Tesla T4


In [3]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

raw = load_dataset("mriadur/phishing-email-dataset", split="train")
df = raw.to_pandas()

df = df.drop(columns=["Unnamed: 0"])
df = df.dropna(subset=["Email Text"])
df = df[~df["Email Text"].str.strip().str.lower().isin(["null", "empty", ""])]
df = df.drop_duplicates(subset=["Email Text"])
df["label"] = df["Email Type"].map({"Safe Email": 0, "Phishing Email": 1})
print("Clean shape:", df.shape)

X = df["Email Text"]
y = df["label"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)
print("Train / Val / Test:", len(X_train), len(X_val), len(X_test))


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Clean shape: (17535, 3)
Train / Val / Test: 14028 1753 1754


In [4]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = Dataset.from_dict({"text": list(X_train), "label": list(y_train)})
val_ds   = Dataset.from_dict({"text": list(X_val),   "label": list(y_val)})
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2)

args = TrainingArguments(
    output_dir="bert_out",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    report_to=[],
    seed=42,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer))

trainer.train()


Map:   0%|          | 0/14028 [00:00<?, ? examples/s]

Map:   0%|          | 0/1753 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.062968,0.046315
2,0.024095,0.059953
3,0.010585,0.058189


TrainOutput(global_step=2631, training_loss=0.04672186649381457, metrics={'train_runtime': 927.7723, 'train_samples_per_second': 45.36, 'train_steps_per_second': 2.836, 'total_flos': 2787379002519552.0, 'train_loss': 0.04672186649381457, 'epoch': 3.0})

In [5]:
import numpy as np
out = trainer.predict(val_ds)
pred = np.argmax(out.predictions, axis=1)

print("Accuracy :", round(accuracy_score(y_val, pred), 4))
print("Precision:", round(precision_score(y_val, pred), 4))
print("Recall   :", round(recall_score(y_val, pred), 4))
print("F1       :", round(f1_score(y_val, pred), 4))
print(confusion_matrix(y_val, pred))


Accuracy : 0.9892
Precision: 0.9907
Recall   : 0.9802
F1       : 0.9854
[[1092    6]
 [  13  642]]


In [6]:
print(df.shape, len(X_train), len(X_val), len(X_test))



(17535, 3) 14028 1753 1754


In [7]:
from scipy.special import softmax
bert_proba = softmax(out.predictions, axis=1)[:, 1]

print("Thresh  Precision  Recall   F1      Missed scams")
for t in [0.30, 0.35, 0.40, 0.45, 0.50]:
    p = (bert_proba >= t).astype(int)
    fn = ((y_val == 1) & (p == 0)).sum()
    print(f"{t:.2f}    {precision_score(y_val, p):.4f}    {recall_score(y_val, p):.4f}  {f1_score(y_val, p):.4f}   {fn}")


Thresh  Precision  Recall   F1      Missed scams
0.30    0.9878    0.9863  0.9870   9
0.35    0.9893    0.9847  0.9870   10
0.40    0.9908    0.9847  0.9877   10
0.45    0.9908    0.9817  0.9862   12
0.50    0.9907    0.9802  0.9854   13


In [8]:
print("Thresh  Precision  Recall   F1      Missed scams")
for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    p = (bert_proba >= t).astype(int)
    fn = ((y_val == 1) & (p == 0)).sum()
    print(f"{t:.2f}    {precision_score(y_val, p):.4f}    {recall_score(y_val, p):.4f}  {f1_score(y_val, p):.4f}   {fn}")


Thresh  Precision  Recall   F1      Missed scams
0.05    0.9878    0.9878  0.9878   8
0.10    0.9878    0.9863  0.9870   9
0.15    0.9878    0.9863  0.9870   9
0.20    0.9878    0.9863  0.9870   9
0.25    0.9878    0.9863  0.9870   9
0.30    0.9878    0.9863  0.9870   9


In [9]:
test_ds = Dataset.from_dict({"text": list(X_test), "label": list(y_test)})
test_ds = test_ds.map(tokenize, batched=True)

out_test = trainer.predict(test_ds)
test_proba = softmax(out_test.predictions, axis=1)[:, 1]
final_pred = (test_proba >= 0.05).astype(int)

print("FINAL TEST RESULTS — DistilBERT @ threshold 0.05")
print("Accuracy :", round(accuracy_score(y_test, final_pred), 4))
print("Precision:", round(precision_score(y_test, final_pred), 4))
print("Recall   :", round(recall_score(y_test, final_pred), 4))
print("F1       :", round(f1_score(y_test, final_pred), 4))
print(confusion_matrix(y_test, final_pred))


Map:   0%|          | 0/1754 [00:00<?, ? examples/s]

FINAL TEST RESULTS — DistilBERT @ threshold 0.05
Accuracy : 0.9903
Precision: 0.9878
Recall   : 0.9863
F1       : 0.987
[[1090    8]
 [   9  647]]


In [10]:
trainer.save_model('distilbert')
tokenizer.save_pretrained('distilbert')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert/tokenizer_config.json', 'distilbert/tokenizer.json')

In [11]:
!zip -r distilbert.zip distilbert


  adding: distilbert/ (stored 0%)
  adding: distilbert/model.safetensors (deflated 8%)
  adding: distilbert/training_args.bin (deflated 54%)
  adding: distilbert/tokenizer.json (deflated 71%)
  adding: distilbert/config.json (deflated 49%)
  adding: distilbert/tokenizer_config.json (deflated 43%)
